# SMARTADDICT v6 - RESIDUAL NEURAL CLASSIFIER (local run)

PyTorch residual-block classifier for the Playground Series S6E8 smartphone-addiction
competition, adapted to run in a local environment.

Changes from the original Kaggle notebook:

1. **Data paths** now point to the local `datasets/` folder (`train.csv`, `test.csv`,
   `sample_submission.csv`) instead of `/kaggle/input/...`.
2. **Device** is auto-selected - CUDA when available, otherwise CPU.
3. **External blend submissions are optional.** The code looks for them inside a
   configured `blend_dir` and skips any that are missing. If no external file is
   available it falls back to a pure-network submission, so the notebook runs
   identically on Kaggle and locally.

Pipeline: median / most-frequent imputation + one-hot encoding inside a scikit-learn
`ColumnTransformer` -> a 128-wide residual MLP (BatchNorm + Dropout) -> AdamW with
BCE-with-logits loss -> validation ROC-AUC model selection -> weighted blend.

In [1]:
# 1. Setup
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

In [2]:
# 2. Define the residual block
class ResidualBlock(nn.Module):
    # Define initialization
    def __init__(self, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define layers
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.BatchNorm1d(hidden_dim)

    # Define forward pass
    def forward(self, features):
        # Save residual
        residual = features

        # Transform features
        features = self.linear1(features)
        features = self.activation(features)
        features = self.dropout(features)
        features = self.linear2(features)

        # Add residual and normalize
        features = features + residual
        features = self.batch_norm(features)
        features = self.activation(features)

        # Return features
        return features


# Define the residual classifier
class ResidualClassifier(nn.Module):
    # Define initialization
    def __init__(self, input_dim, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define model layers
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
        )
        self.output_layer = nn.Linear(hidden_dim, 1)

    # Define forward pass
    def forward(self, features):
        # Transform features
        features = self.input_layer(features)
        features = self.activation(features)
        features = self.dropout(features)
        features = self.blocks(features)

        # Return one logit per row
        return self.output_layer(features).squeeze(1)

In [3]:
# 3. Training and inference helpers
# Set random seeds
def set_seed(seed):
    # Seed all random generators
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Seed CUDA when available
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Create the preprocessing pipeline
def create_preprocessor(features):
    # Detect categorical columns
    categorical_columns = features.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    # Detect numerical columns
    numerical_columns = features.select_dtypes(
        exclude=["object", "category"]
    ).columns.tolist()

    # Define numerical preprocessing
    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ]
    )

    # Define categorical preprocessing
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    # Return combined preprocessing
    return ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_columns),
            ("cat", categorical_transformer, categorical_columns),
        ]
    )


# Prepare train, validation, and test matrices
def preprocess_data(train_df, test_df, target_column, seed):
    # Separate predictors and target
    features = train_df.drop(columns=["id", target_column])
    target = train_df[target_column].to_numpy(dtype=np.float32)
    test_features = test_df.drop(columns=["id"])

    # Split before fitting preprocessing to prevent leakage
    train_features, valid_features, train_target, valid_target = train_test_split(
        features,
        target,
        test_size=0.1,
        random_state=seed,
        stratify=target,
    )

    # Fit preprocessing on the training fold only
    preprocessor = create_preprocessor(train_features)
    train_matrix = preprocessor.fit_transform(train_features)
    valid_matrix = preprocessor.transform(valid_features)
    test_matrix = preprocessor.transform(test_features)

    # Convert matrices to float32
    train_matrix = np.asarray(train_matrix, dtype=np.float32)
    valid_matrix = np.asarray(valid_matrix, dtype=np.float32)
    test_matrix = np.asarray(test_matrix, dtype=np.float32)

    # Return prepared data
    return (
        train_matrix,
        valid_matrix,
        train_target,
        valid_target,
        test_matrix,
        test_df["id"].copy(),
    )


# Create a tensor data loader
def create_loader(features, target=None, batch_size=4096, shuffle=False):
    # Convert features to a tensor
    feature_tensor = torch.from_numpy(features)

    # Create an inference loader
    if target is None:
        return DataLoader(
            feature_tensor,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=torch.cuda.is_available(),
        )

    # Convert target to a tensor
    target_tensor = torch.from_numpy(target)

    # Create a supervised loader
    dataset = TensorDataset(feature_tensor, target_tensor)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=torch.cuda.is_available(),
    )


# Predict positive-class probabilities
def predict_probabilities(model, loader, device):
    # Set evaluation mode
    model.eval()

    # Initialize predictions
    predictions = []

    # Disable gradient tracking
    with torch.no_grad():
        # Iterate batches
        for batch in loader:
            # Support supervised and inference loaders
            batch_features = batch[0] if isinstance(batch, list) else batch
            batch_features = batch_features.to(device, non_blocking=True)

            # Convert logits to probabilities
            logits = model(batch_features)
            probabilities = torch.sigmoid(logits)
            predictions.append(probabilities.cpu().numpy())

    # Concatenate predictions
    return np.concatenate(predictions)


# Train the classifier
def train_model(
    model,
    train_loader,
    valid_loader,
    valid_target,
    device,
    epochs,
    learning_rate,
):
    # Define binary classification loss
    criterion = nn.BCEWithLogitsLoss()

    # Define optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-3,
    )

    # Initialize best validation state
    best_auc = -np.inf
    best_state = None

    # Run training epochs
    for epoch in range(epochs):
        # Set training mode
        model.train()

        # Initialize loss totals
        total_loss = 0.0
        total_rows = 0

        # Iterate training batches
        for batch_features, batch_target in train_loader:
            # Move batch to device
            batch_features = batch_features.to(device, non_blocking=True)
            batch_target = batch_target.to(device, non_blocking=True)

            # Update model weights
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = criterion(logits, batch_target)
            loss.backward()
            optimizer.step()

            # Accumulate row-weighted loss
            total_loss += loss.item() * batch_features.size(0)
            total_rows += batch_features.size(0)

        # Evaluate validation ROC AUC
        valid_predictions = predict_probabilities(
            model,
            valid_loader,
            device,
        )
        valid_auc = roc_auc_score(valid_target, valid_predictions)

        # Print epoch metrics
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train BCE: {total_loss / total_rows:.6f} | "
            f"Valid AUC: {valid_auc:.6f}"
        )

        # Save an independent copy of the best weights
        if valid_auc > best_auc:
            best_auc = valid_auc
            best_state = copy.deepcopy(model.state_dict())

    # Restore best weights
    model.load_state_dict(best_state)

    # Print best score
    print(f"Best validation AUC: {best_auc:.6f}")

    # Return trained model
    return model


# Blend predictions and save the submission
def blend_predictions(
    test_ids,
    prediction_configs,
    target_column,
    output_path,
):
    # Initialize blend totals
    weighted_predictions = np.zeros(len(test_ids), dtype=np.float64)
    total_weight = 0.0

    # Blend every prediction source
    for prediction_name, config in prediction_configs.items():
        # Extract predictions and weight
        predictions = np.asarray(config["predictions"], dtype=np.float64)
        weight = float(config["weight"])

        # Validate prediction length
        if len(predictions) != len(test_ids):
            raise ValueError(
                f"{prediction_name} has {len(predictions)} rows; "
                f"expected {len(test_ids)}."
            )

        # Add weighted predictions
        weighted_predictions += predictions * weight
        total_weight += weight

        # Print blend information
        print(f"Blending {prediction_name} | Weight: {weight}")

    # Validate total weight
    if total_weight <= 0.0:
        raise ValueError("The total blend weight must be positive.")

    # Compute final probabilities
    final_predictions = weighted_predictions / total_weight

    # Create submission frame
    submission = pd.DataFrame(
        {
            "id": test_ids.to_numpy(),
            target_column: np.clip(final_predictions, 0.0, 1.0),
        }
    )

    # Save submission
    submission.to_csv(output_path, index=False)

    # Print confirmation
    print(f"Submission saved to {output_path}")

In [ ]:
# 4. Define the main function
def main():
    # Define configuration
    target_column = "addicted_label"
    data_dir = Path("datasets")
    # Folder for optional external submission files to blend with the network.
    # Only files that actually exist are used; missing ones are skipped.
    blend_dir = Path("datasets") / "vault"
    output_path = Path("submission.csv")
    seed = 42

    # Set random seeds
    set_seed(seed)

    # Load competition data
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")

    # Validate expected schemas
    expected_test_columns = set(train_df.columns) - {target_column}
    if set(test_df.columns) != expected_test_columns:
        raise ValueError("Train and test feature columns do not match.")

    # Prepare data
    (
        train_matrix,
        valid_matrix,
        train_target,
        valid_target,
        test_matrix,
        test_ids,
    ) = preprocess_data(
        train_df,
        test_df,
        target_column,
        seed,
    )

    # Create loaders
    train_loader = create_loader(
        train_matrix,
        train_target,
        batch_size=4096,
        shuffle=True,
    )
    valid_loader = create_loader(
        valid_matrix,
        batch_size=8192,
    )
    test_loader = create_loader(
        test_matrix,
        batch_size=8192,
    )

    # Select compute device
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"Using device: {device}")
    print(f"Processed feature count: {train_matrix.shape[1]}")

    # Create classifier
    model = ResidualClassifier(
        input_dim=train_matrix.shape[1],
        hidden_dim=128,
        dropout=0.2,
    ).to(device)

    # Train classifier
    model = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        valid_target=valid_target,
        device=device,
        epochs=24,
        learning_rate=1e-3,
    )

    # Predict test probabilities
    test_predictions = predict_probabilities(
        model,
        test_loader,
        device,
    )

    # Discover optional external submission files to blend with.
    prediction_configs = {}
    candidate_blends = [
        ("sub1", blend_dir / "submission.csv", 2.9),
        ("sub2", blend_dir / "submission (1).csv", 0.1),
    ]

    for name, blend_path, weight in candidate_blends:
        if not blend_path.exists():
            print(f"External predictions not found (skipped): {name} -> {blend_path}")
            continue

        # Load the external submission
        external = pd.read_csv(blend_path)

        # Validate external submission ids
        if not np.array_equal(external["id"].to_numpy(), test_ids.to_numpy()):
            print(
                f"Warning: ids in {name} ({blend_path}) do not match test.csv; "
                "external file skipped."
            )
            continue

        # Register the external submission
        prediction_configs[name] = {
            "predictions": external[target_column].to_numpy(),
            "weight": weight,
        }
        print(f"Found external predictions: {name} -> {blend_path}")

    # Always add the network as a blend source. When no external submission is
    # available, the network alone defines the final prediction.
    prediction_configs["nn"] = {
        "predictions": test_predictions,
        "weight": 1.0 if not prediction_configs else 1e-6,
    }

    # Blend predictions and save the submission
    blend_predictions(
        test_ids=test_ids,
        prediction_configs=prediction_configs,
        target_column=target_column,
        output_path=output_path,
    )


# Call the main function
if __name__ == "__main__":
    main()

Using device: cuda
Processed feature count: 26
Epoch 01 | Train BCE: 0.352622 | Valid AUC: 0.930712
Epoch 02 | Train BCE: 0.301292 | Valid AUC: 0.933462
Epoch 03 | Train BCE: 0.293771 | Valid AUC: 0.935571


In [ ]:
print("All the things are setup")